# Text2ToBI — Run Summary

Scans a `runs/` directory tree, parses every `hparams*.json` it finds,
and exports a formatted `.xlsx` summary table.

**Usage:** Edit `RUNS_ROOT` and `OUTPUT_PATH` in Cell 1, then run all cells.
Call `build_summary(sort_by=..., ascending=False)` to re-sort without re-scanning.


In [ ]:
# 1. Zip the folder directly into Colab's temporary local storage
!zip -r /content/my_files.zip /content/drive/MyDrive/Capstone/project/runs

# 2. (Optional) If the folder is outside your main Drive, just check the path in the left sidebar

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Configuration                                      ║
# ╚══════════════════════════════════════════════════════════════╝

RUNS_ROOT   = "/content/drive/MyDrive/Capstone/project/runs"
OUTPUT_PATH = f"{RUNS_ROOT}/runs_summary.xlsx"

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

print("✓ Config loaded.")
print(f"  Scanning : {RUNS_ROOT}")
print(f"  Output   : {OUTPUT_PATH}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import json, math, os, re
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import (
    Alignment, Font, PatternFill, Border, Side
)
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter

print("✓ Imports OK.")


In [ ]:
import os
# List what the scanner actually sees
for root, dirs, files in os.walk(RUNS_ROOT):
    for f in files:
        if 'hparams' in f.lower():
            print(os.path.join(root, f))

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Folder-name parser                                 ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Folder naming convention (from project spec):
#
#   corpus  : libri | sbc | c100 | c360 | all | libri+sbc | libri+sbc+bu | ...
#   loss    : weighted | stl          (default: stl)
#   punct   : pp                      (default: absent → no punct stripping)
#   pos     : posonly | pos           (default: absent → text only)
#   lr      : lr<value>               e.g. lr1e4, lr1e5
#   warmup  : w<steps> | warm<steps>  e.g. w500, warm500, warmup500

import re

def _norm_lr(raw):
    raw = raw.lower().replace("_", "")
    if re.match(r"^\d+e-\d+$", raw):
        return raw
    m = re.match(r"^(\d+)e0*(\d+)$", raw)
    return f"{m.group(1)}e-{m.group(2)}" if m else raw

def parse_folder_name(run_folder_name, parent_folder_name=""):
    cfg = {
        "corpus":  None,
        "loss":    "stl",
        "punct":   "—",
        "pos":     "—",
        "lr":      "—",
        "warmup":  "—",
    }

    # ── Corpus ────────────────────────────────────────────────────
    if parent_folder_name and parent_folder_name.lower() not in ("runs", "full", ""):
        cfg["corpus"] = parent_folder_name.lower()
    elif parent_folder_name.lower() == "full":
        cfg["corpus"] = "full"
    if not cfg["corpus"]:
        m = re.search(
            r"((?:libri|sbc|c100|c360|all|bu)(?:[+_](?:libri|sbc|c100|c360|all|bu))*)",
            run_folder_name.lower()
        )
        cfg["corpus"] = m.group(1).replace("_", "+") if m else "unknown"

    name = run_folder_name.lower()

    # ── Loss ──────────────────────────────────────────────────────
    cfg["loss"] = "weighted" if "weighted" in name else "stl"

    # ── Punctuation ───────────────────────────────────────────────
    if re.search(r"(?<![a-z])pp(?![a-z])", name):
        cfg["punct"] = "yes"

    # ── POS ───────────────────────────────────────────────────────
    # Check posonly first (more specific), then pos (standalone)
    if "posonly" in name or "pos_only" in name:
        cfg["pos"] = "posonly"
    elif re.search(r"(?<![a-z])pos(?![a-z])", name):
        cfg["pos"] = "pos"

    # ── Learning rate ─────────────────────────────────────────────
    lr_m = re.search(r"lr([0-9]+e[-_]?[0-9]+)", name)
    if lr_m:
        cfg["lr"] = _norm_lr(lr_m.group(1))

    # ── Warmup ────────────────────────────────────────────────────
    wu_m = re.search(r"(?:warm(?:up)?|(?<![a-z])w)(\d+)", name)
    if wu_m:
        cfg["warmup"] = wu_m.group(1)

    return cfg


# ── Unit tests ────────────────────────────────────────────────────
_tests = [
    ("libri+sbc_posonly_stl",  "libri+sbc", {"corpus": "libri+sbc", "loss": "stl",      "pos": "posonly", "warmup": "—",   "lr": "—"}),
    ("libri+sbc_pos_stl",      "libri+sbc", {"corpus": "libri+sbc", "loss": "stl",      "pos": "pos",     "warmup": "—",   "lr": "—"}),
    ("libri+sbc_stl",          "libri+sbc", {"corpus": "libri+sbc", "loss": "stl",      "pos": "—",       "warmup": "—",   "lr": "—"}),
    ("libri+sbc_weighted",     "libri+sbc", {"corpus": "libri+sbc", "loss": "weighted", "pos": "—",       "warmup": "—",   "lr": "—"}),
    ("libri+sbc_stl_lr1e4",    "libri+sbc", {"corpus": "libri+sbc", "loss": "stl",      "pos": "—",       "warmup": "—",   "lr": "1e-4"}),
    ("libri+sbc_stl_warmup500","libri+sbc", {"corpus": "libri+sbc", "loss": "stl",      "pos": "—",       "warmup": "500", "lr": "—"}),
    ("sbc_stl",                "sbc",       {"corpus": "sbc",        "loss": "stl",      "pos": "—"}),
    ("libri_weighted",         "libri",     {"corpus": "libri",      "loss": "weighted", "pos": "—"}),
]
for run_name, parent, expected in _tests:
    result = parse_folder_name(run_name, parent)
    for k, v in expected.items():
        assert result[k] == v, f"FAIL {run_name!r}: {k}={result[k]!r}, expected {v!r}"
print("✓ Cell 3: folder-name parser OK — all unit tests passed.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — hparams.json scanner                               ║
# ╚══════════════════════════════════════════════════════════════╝

# hparams files may be named:
#   hparams.json  |  hparams1.json  |  <run_name>_hparams.json  etc.
# Rule: match any file whose name ends with  hparams[optional_suffix].json
_HPARAMS_RE = re.compile(r"hparams\d*\.json$", re.IGNORECASE)

def _nan_to_dash(v):
    """Return "—" for NaN/None, else round float to 4dp string."""
    if v is None:
        return "—"
    if isinstance(v, float):
        if math.isnan(v):
            return "—"
        return round(v, 4)
    return v

def _best_epoch(history_list):
    """Return 1-based epoch index of the max value in a list, or "—"."""
    if not history_list:
        return "—"
    valid = [(i, v) for i, v in enumerate(history_list)
             if isinstance(v, float) and not math.isnan(v)]
    if not valid:
        return "—"
    best_i = max(valid, key=lambda x: x[1])[0]
    return best_i + 1   # 1-based

def scan_runs(runs_root):
    """
    Walk runs_root recursively and collect one row per hparams file found.

    Raises FileNotFoundError if runs_root does not exist.
    Raises ValueError if a hparams file cannot be parsed.
    """
    root = Path(runs_root)
    if not root.exists():
        raise FileNotFoundError(f"RUNS_ROOT not found: {runs_root}")

    rows = []

    for hparams_path in sorted(root.rglob("*")):
        if not hparams_path.is_file():
            continue
        if not _HPARAMS_RE.search(hparams_path.name):
            continue

        # ── Load JSON ────────────────────────────────────────────
        try:
            with open(hparams_path, encoding="utf-8") as fh:
                h = json.load(fh)
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON parse error in {hparams_path}: {e}")

        # ── Resolve run and parent folder names ───────────────────
        run_folder   = hparams_path.parent.name
        parent_folder = hparams_path.parent.parent.name

        cfg = parse_folder_name(run_folder, parent_folder)

        # ── Extract metrics ───────────────────────────────────────
        res  = h.get("results", {})
        test = res.get("test", {})
        b    = test.get("boundary",  {})
        i_   = test.get("intonation", {})
        x    = test.get("break_idx",  {})
        vh   = res.get("val_history",   {})
        th   = res.get("train_history", {})
        trn  = h.get("training", {})

        row = {
            # Identification
            "RUN":               run_folder,
            # Config (parsed from folder name)
            "CORPUS":            cfg["corpus"],
            "LOSS":              cfg["loss"],
            "POS":               cfg["pos"],
            "LR":                cfg["lr"] if cfg["lr"] != "—"
                                           else str(trn.get("learning_rate", "—")),
            "WARMUP":            cfg["warmup"] if cfg["warmup"] != "—"
                                              else (str(trn.get("warmup_steps", "—"))
                                                    if trn.get("warmup_steps", 0) > 0 else "—"),
            # Best epoch per head (val)
            "BEST_EPOCH_B":      _best_epoch(vh.get("boundary_f1",    [])),
            "BEST_EPOCH_I":      _best_epoch(vh.get("intonation_f1",  [])),
            "BEST_EPOCH_X":      _best_epoch(vh.get("break_idx_f1",   [])),
            # Test metrics — boundary
            "TEST_B_F1":         _nan_to_dash(b.get("f1")),
            "TEST_B_P":          _nan_to_dash(b.get("precision")),
            "TEST_B_R":          _nan_to_dash(b.get("recall")),
            # Test metrics — intonation
            "TEST_I_F1":         _nan_to_dash(i_.get("f1")),
            "TEST_I_P":          _nan_to_dash(i_.get("precision")),
            "TEST_I_R":          _nan_to_dash(i_.get("recall")),
            # Test metrics — break index
            "TEST_X_F1":         _nan_to_dash(x.get("f1")),
            "TEST_X_P":          _nan_to_dash(x.get("precision")),
            "TEST_X_R":          _nan_to_dash(x.get("recall")),
            # Best val boundary F1
            "VAL_B_F1":          _nan_to_dash(res.get("best_val_boundary_f1")),
            # Best train boundary F1 (epoch matching best val epoch)
            "TRAIN_B_F1":        _nan_to_dash(
                                     th.get("boundary_f1", [None] * 6)[
                                         (_best_epoch(vh.get("boundary_f1", [])) - 1)
                                         if isinstance(_best_epoch(vh.get("boundary_f1", [])), int)
                                         else 0
                                     ]
                                 ),
        }
        rows.append(row)

    if not rows:
        raise FileNotFoundError(
            f"No hparams*.json files found under {runs_root}. "
            "Check RUNS_ROOT and folder structure."
        )

    return rows

print("✓ Cell 4: scanner defined.")
print("  scan_runs(runs_root) → list of row dicts")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — xlsx writer                                        ║
# ╚══════════════════════════════════════════════════════════════╝

# Column definitions: (key, header label, width)
COLUMNS = [
    ("RUN",          "RUN",            32),
    ("CORPUS",       "CORPUS",         14),
    ("LOSS",         "LOSS",            9),
    ("POS",          "POS",            10),
    ("LR",           "LR",             10),
    ("WARMUP",       "WARMUP",          8),
    ("BEST_EPOCH_B", "EPOCH_B",         8),
    ("BEST_EPOCH_I", "EPOCH_I",         8),
    ("BEST_EPOCH_X", "EPOCH_X",         8),
    ("TEST_B_F1",    "B_F1 ▼",         10),   # primary sort column
    ("TEST_B_P",     "B_PREC",          9),
    ("TEST_B_R",     "B_REC",           9),
    ("TEST_I_F1",    "I_F1",            9),
    ("TEST_I_P",     "I_PREC",          9),
    ("TEST_I_R",     "I_REC",           9),
    ("TEST_X_F1",    "X_F1",            9),
    ("TEST_X_P",     "X_PREC",          9),
    ("TEST_X_R",     "X_REC",           9),
    ("VAL_B_F1",     "VAL_B_F1",       11),
    ("TRAIN_B_F1",   "TRAIN_B_F1",     12),
]

_HEADER_FILL  = PatternFill("solid", start_color="1F4E79")   # dark blue
_HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
_DATA_FONT    = Font(name="Arial", size=10)
_ALT_FILL     = PatternFill("solid", start_color="EBF3FB")   # light blue
_BORDER_SIDE  = Side(style="thin", color="CCCCCC")
_CELL_BORDER  = Border(
    left=_BORDER_SIDE, right=_BORDER_SIDE,
    top=_BORDER_SIDE,  bottom=_BORDER_SIDE
)
_CENTER       = Alignment(horizontal="center", vertical="center")
_LEFT         = Alignment(horizontal="left",   vertical="center")

def write_xlsx(rows, output_path, sort_by="TEST_B_F1", ascending=False):
    """
    Write rows to an xlsx file.

    Parameters
    ----------
    rows       : list of dicts from scan_runs()
    output_path: str  — full path including filename
    sort_by    : str  — column key to sort by (default: TEST_B_F1)
    ascending  : bool — sort direction (default: False = best first)
    """
    # ── Sort ──────────────────────────────────────────────────────
    def _sort_key(row):
        v = row.get(sort_by, "—")
        if v == "—" or v is None:
            return (-1 if not ascending else 9999)
        return v

    sorted_rows = sorted(rows, key=_sort_key, reverse=not ascending)

    # ── Workbook ──────────────────────────────────────────────────
    wb = Workbook()
    ws = wb.active
    ws.title = "Run Summary"
    ws.freeze_panes = "B2"   # freeze row 1 (header) + col A (run name)

    # ── Header row ────────────────────────────────────────────────
    for col_idx, (key, label, width) in enumerate(COLUMNS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=label)
        cell.font      = _HEADER_FONT
        cell.fill      = _HEADER_FILL
        cell.alignment = _CENTER
        cell.border    = _CELL_BORDER
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.row_dimensions[1].height = 20

    # ── Data rows ─────────────────────────────────────────────────
    for row_idx, row in enumerate(sorted_rows, start=2):
        is_alt = (row_idx % 2 == 0)
        for col_idx, (key, label, _) in enumerate(COLUMNS, start=1):
            val  = row.get(key, "—")
            c    = ws.cell(row=row_idx, column=col_idx, value=val)
            c.font   = _DATA_FONT
            c.border = _CELL_BORDER
            c.alignment = _LEFT if col_idx == 1 else _CENTER
            if is_alt and val != "—":
                c.fill = _ALT_FILL
            # 4dp for floats
            if isinstance(val, float):
                c.number_format = "0.0000"

    # ── Conditional formatting on TEST_B_F1 only ─────────────────
    # Find the column letter for TEST_B_F1
    b_f1_col = next(
        get_column_letter(i + 1)
        for i, (key, _, __) in enumerate(COLUMNS)
        if key == "TEST_B_F1"
    )
    n_rows = len(sorted_rows)
    b_f1_range = f"{b_f1_col}2:{b_f1_col}{n_rows + 1}"

    ws.conditional_formatting.add(
        b_f1_range,
        ColorScaleRule(
            start_type="min",  start_color="F8696B",   # red   — low F1
            mid_type="num",    mid_value=0.75,
                               mid_color="FFEB84",     # yellow — mid
            end_type="max",    end_color="63BE7B",     # green  — high F1
        )
    )

    # ── Save ──────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    wb.save(output_path)
    print(f"✓ Saved {n_rows} runs → {output_path}")
    print(f"  Sorted by: {sort_by}  ({'asc' if ascending else 'desc'})")


print("✓ Cell 5: xlsx writer defined.")
print("  write_xlsx(rows, output_path, sort_by=..., ascending=False)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — build_summary()  ← call this cell to run          ║
# ╚══════════════════════════════════════════════════════════════╝

def build_summary(sort_by="TEST_B_F1", ascending=False):
    """
    Scan RUNS_ROOT, parse all hparams files, and write summary xlsx.

    Parameters
    ----------
    sort_by   : column key to sort by.
                Valid keys: RUN, CORPUS, LOSS, POS, LR, WARMUP,
                BEST_EPOCH_B, BEST_EPOCH_I, BEST_EPOCH_X,
                TEST_B_F1, TEST_B_P, TEST_B_R,
                TEST_I_F1, TEST_I_P, TEST_I_R,
                TEST_X_F1, TEST_X_P, TEST_X_R,
                VAL_B_F1, TRAIN_B_F1
    ascending : sort direction (False = best first)

    Examples
    --------
    build_summary()                               # default: test boundary F1 desc
    build_summary(sort_by="TEST_I_F1")            # sort by intonation F1
    build_summary(sort_by="CORPUS", ascending=True)
    build_summary(sort_by="VAL_B_F1")
    """
    print(f"Scanning {RUNS_ROOT} ...")
    rows = scan_runs(RUNS_ROOT)
    print(f"Found {len(rows)} run(s).")
    write_xlsx(rows, OUTPUT_PATH, sort_by=sort_by, ascending=ascending)

# ── Run immediately ───────────────────────────────────────────────
build_summary()
